In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.cluster import HDBSCAN
from sklearn.decomposition import PCA

class Clustering(object):

    def __init__(self, min_cluster_size: int, min_samples: int, cluster_selection_epsilon: float, 
                 max_cluster_size: int, 
                 metric: str, alpha: float, p: str, cluster_selection_method: str, allow_single_cluster: bool,
                 gen_min_span_tree: bool, verbose: bool, output_type: str, prediction_data: bool, build_algo: str,
                 build_kwds: str, device_ids: str,
                 plotting_x: str, plotting_y: str,):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        self.cluster_selection_epsilon = cluster_selection_epsilon
        self.max_cluster_size = max_cluster_size
        self.metric = metric
        self.alpha = alpha
        self.p = p
        self.cluster_selection_method = cluster_selection_method
        self.allow_single_cluster = allow_single_cluster
        self.gen_min_span_tree = gen_min_span_tree
        self.verbose = verbose
        self.output_type = output_type
        self.prediction_data = prediction_data
        self.build_algo = build_algo
        self.build_kwds = build_kwds
        self.device_ids = device_ids
        self.plotting_x = plotting_x
        self.plotting_y = plotting_y

    def df_read(self):
        global df_pandas_global
        
        df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                                date_format = {'Date': '%d/%m/%Y', 'Time': '%H:%M:%S'},
                                dayfirst = True)
        df_pandas_global = df_pandas
    
    def to_cudf(self):
        global df_cudf_global
        
        df_cudf = cudf.DataFrame(df_pandas_global).copy()
        df_cudf_global = df_cudf
        
    def converting(self):
        global df_cudf_converted_global
        
        df_cudf_converted_global = df_cudf_global.astype(float).copy()

    def HDBSCAN(self):
        global HDBSCAN_global
        global labels_global
        global cluster_centers_global

        HDBSCAN_float = cuml.cluster.hdbscan.HDBSCAN(min_cluster_size=self.min_cluster_size, min_samples=self.min_samples, 
                                                     cluster_selection_epsilon=self.cluster_selection_epsilon, 
                                                     max_cluster_size=self.max_cluster_size, 
                                                     metric=self.metric, alpha=self.alpha, p=self.p, 
                                                     cluster_selection_method=self.cluster_selection_method, 
                                                     allow_single_cluster=self.allow_single_cluster, 
                                                     gen_min_span_tree=self.gen_min_span_tree, 
                                                     verbose=self.verbose, output_type=self.output_type, 
                                                     prediction_data=self.prediction_data, build_algo=self.build_algo, 
                                                     build_kwds=self.build_kwds, device_ids=self.device_ids)
        HDBSCAN = HDBSCAN_float.fit(df_cudf_converted_global)
        HDBSCAN_global = HDBSCAN

        labels = HDBSCAN_float.labels_
        labels_global = cudf.DataFrame(labels, columns=['Labels'])

    def labels_concat(self):
        global df_concat_global
        global df_concat_pandas_global

        df_concat = cudf.concat([df_cudf_converted_global, labels_global], axis=1)
        df_concat_global = df_concat

        df_concat_pandas = df_concat.to_pandas()
        df_concat_pandas_global = df_concat_pandas

    def plotting(self):


        sns.scatterplot(data=df_concat_pandas_global, x=self.plotting_x, y=self.plotting_y, hue='Labels', palette='Set1')
        plt.show()
        
        
    def main(self):
        self.df_read()
        self.to_cudf()
        self.converting()
        st = time.time()
        self.HDBSCAN()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.labels_concat()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.plotting()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        